In [0]:
##DimUser

In [0]:
file_name = dbutils.widgets.get("file_name")

In [0]:
file_name

In [0]:
Dim_df  = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","parquet") \
        .option("inferSchema","True") \
            .option("cloudFiles.schemaLocation",f"abfss://silver@spotifystorageaccount02.dfs.core.windows.net/{file_name}/check_point") \
                .load(f"abfss://bronze@spotifystorageaccount02.dfs.core.windows.net/{file_name}")

In [0]:
import os 

print(os.getcwd())

In [0]:
print(os.path)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


import os 
import sys


##project_pth = os.path.join(os.getcwd(),'..','..')
##sys.path.append(project_pth)


utils_path = "/Workspace/Users/syammangat2001@gmail.com/SpotifyAzureProject/Spotify_DAB_bundles/utils"


if utils_path not in sys.path:
    sys.path.append(utils_path)

    
print(sys.path)

In [0]:
print(sys.path)

In [0]:
 from Transformations import reusable
 
 df_user_obj = reusable()


In [0]:

df_user = df_user_obj.dropColumns(Dim_df,['_rescued_data'])
display(df_user)

In [0]:
if f"{file_name}" == "DimUser":
    df_user = df_user.dropDuplicates(['user_id'])

elif f"{file_name}" == "DimArtist":
    df_user = df_user.dropDuplicates(['artist_id'])

elif f"{file_name}" == "DimDate":
    df_user = df_user.dropDuplicates(['date_key'])

elif f"{file_name}" == "DimTrack":
    df_user = df_user.dropDuplicates(['track_id'])
    df_user = df_user.withColumn("duration_flg",when(col("duration_sec")<150,"low") \
        .when(col('duration_sec')<300,"medium") \
            .otherwise("high"))
    df_user = df_user.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))


In [0]:
display(df_user)

In [0]:
df_user.writeStream.format("delta") \
    .outputMode("append") \
        .option("checkpointLocation",f"abfss://silver@spotifystorageaccount02.dfs.core.windows.net/{file_name}/check_point") \
            .trigger(once=True) \
                .option("path",f"abfss://silver@spotifystorageaccount02.dfs.core.windows.net/{file_name}/data") \
                    .toTable(f"dev_catalog.silver.{file_name}")


In [0]:
%sql

          select * from dev_catalog.silver.